In [1]:
import os
from groq import Groq

client = Groq(api_key = os.environ.get("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(f"--- {model.id}")

--- openai/gpt-oss-20b
--- openai/gpt-oss-safeguard-20b
--- meta-llama/llama-prompt-guard-2-22m
--- whisper-large-v3-turbo
--- openai/gpt-oss-120b
--- canopylabs/orpheus-arabic-saudi
--- groq/compound
--- qwen/qwen3.8-27b
--- allam-2-7b
--- meta-llama/llama-prompt-guard-2-86m
--- qwen/qwen3.6-27b
--- groq/compound-mini
--- canopylabs/orpheus-v1-english
--- whisper-large-v3


In [2]:
MODEL = "openai/gpt-oss-20b"

In [3]:
print(model)

Model(id='whisper-large-v3', created=1693721698, object='model', owned_by='OpenAI', active=True, context_window=448, public_apps=None, max_completion_tokens=448, hugging_face_id='openai/whisper-large-v3', name='Whisper', input_modalities=['audio'], output_modalities=['transcription'], context_length=448, max_output_length=448)


In [4]:
memory = []

print("Memory Created")
print(f"Messages stored: {len(memory)}")

Memory Created
Messages stored: 0


In [5]:
def chat(question):
    # Adding user question to the memory.
    memory.append({
        "role": "user",
        "content": question
    })

    # Send full memory to LLM
    response = client.chat.completions.create(
        model = MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a fraud analyst assistant at a bank."
            }
        ] + memory, 
        temperature=0.0,
        max_tokens=200
    )

    answer = response.choices[0].message.content

    memory.append({
        "role": "assistant",
        "content": answer
    })

    return answer

print("Chat function ready")

Chat function ready


In [6]:
memory = []

print("=== Turn 1 ===")
answer1 = chat(
    "Analyze this transaction: $8,500 wire transfer "
    "at 3am, customer avg spend $95"
)

print(f"Assistant: {answer1}")
print(f"Messages in memory: {len(memory)}")

=== Turn 1 ===
Assistant: **Transaction Summary**

| Item | Detail |
|------|--------|
| **Amount** | $8,500.00 |
| **Transfer type** | Wire (domestic/international?) |
| **Time** | 03:00 AM (local time) |
| **Customer profile** | Average spend ≈ $95 per transaction |
| **Frequency** | 1‑time (as reported) |
| **Account age** | (not provided – assume >6 months) |
| **Account status**
Messages in memory: 2


In [7]:
print("=== Turn 2 ===")
answer2 = chat("What was the transaction amount?")

print(f"Assistant: {answer2}")
print(f"Messages in memory: {len(memory)}")

=== Turn 2 ===
Assistant: The transaction amount was **$8,500.00**.
Messages in memory: 4


In [8]:
# Turn 3 — references Turn 1 and 2
print("\n=== Turn 3 ===")
answer3 = chat("What action should the fraud team take?")
print(f"Assistant: {answer3}")
print(f"Messages in memory: {len(memory)}")


=== Turn 3 ===
Assistant: **Recommended Action Plan for the Fraud Team**

| Step | Action | Rationale | Expected Outcome |
|------|--------|-----------|------------------|
| **1. Flag & Hold** | Immediately place a temporary hold on the wire transfer. | The amount ($8,500) is **~90×** the customer’s average spend ($95) and the transfer occurred at 3 AM, a time when legitimate activity is rare. | Prevents potential loss while investigation proceeds. |
| **2. Verify Sender Identity** | • Confirm the customer’s identity via 2‑factor authentication (e.g., OTP, biometric).  <br>• Cross‑check the account holder’s name, address, and contact details against the bank’s records. | Ensures the transaction was authorized by the legitimate account holder.
Messages in memory: 6


In [9]:
print("=== What's Inside Memory ===")
print(f"Total messages: {len(memory)}\n")

for i, message in enumerate(memory):
    role = message["role"].upper()
    content = message["content"][:80]
    print(f"Message {i+1} [{role}]: {content}...")
    print()

=== What's Inside Memory ===
Total messages: 6

Message 1 [USER]: Analyze this transaction: $8,500 wire transfer at 3am, customer avg spend $95...

Message 2 [ASSISTANT]: **Transaction Summary**

| Item | Detail |
|------|--------|
| **Amount** | $8,5...

Message 3 [USER]: What was the transaction amount?...

Message 4 [ASSISTANT]: The transaction amount was **$8,500.00**....

Message 5 [USER]: What action should the fraud team take?...

Message 6 [ASSISTANT]: **Recommended Action Plan for the Fraud Team**

| Step | Action | Rationale | Ex...



In [10]:
# Show the growing problem
print("=== Buffer Memory Growth Problem ===\n")

print("After  1 turn:   2 messages in context")
print("After  5 turns: 10 messages in context")
print("After 10 turns: 20 messages in context")
print("After 20 turns: 40 messages in context")
print("After 50 turns: 100 messages in context")
print("\nEach message can be 200-300 tokens")
print("50 turns = potentially 30,000 tokens")
print("LLM context limit = 8,000 tokens")
print("\nResult: Context window FULL → LLM crashes")

=== Buffer Memory Growth Problem ===

After  1 turn:   2 messages in context
After  5 turns: 10 messages in context
After 10 turns: 20 messages in context
After 20 turns: 40 messages in context
After 50 turns: 100 messages in context

Each message can be 200-300 tokens
50 turns = potentially 30,000 tokens
LLM context limit = 8,000 tokens

Result: Context window FULL → LLM crashes


In [11]:
# Window Memory function
window_memory = []
WINDOW_SIZE = 4

def chat_window(question):
    window_memory.append({
        "role": "user",
        "content": question
    })

    windowed = window_memory[-WINDOW_SIZE:]

    response = client.chat.completions.create(
        model = MODEL,
        messages = [
            {
                "role": "system",
                "content": "You are a fraud analyst assistant."
            }
        ] + windowed,
        temperature = 0.0,
        max_tokens = 150
    )

    answer = response.choices[0].message.content
    window_memory.append({
        "role": "assistant",
        "content": answer
    })

    return answer

print("Window memory ready")
print(f"Window size: {WINDOW_SIZE} messages = {WINDOW_SIZE//2} turns")

Window memory ready
Window size: 4 messages = 2 turns


In [15]:
# Test window memory
# Turn 5 asks about Turn 1 — will it remember?

print("=== Window Memory Test ===\n")

# Turn 1 — introduce the transaction
print("[Turn 1]")
r1 = chat_window(
    "Analyze TXN-W001: $8,500 wire transfer "
    "at 3am, avg spend $95"
)
print(f"Assistant: {r1[:150]}...")
print(f"Memory size: {len(window_memory)} messages\n")

# Turn 2
print("[Turn 2]")
r2 = chat_window("What is the risk level?")
print(f"Assistant: {r2[:150]}...")
print(f"Memory size: {len(window_memory)} messages\n")

# Turn 3 — Turn 1 gets dropped here
print("[Turn 3]")
r3 = chat_window("Now analyze TXN-W002: $45 Starbucks 8am")
print(f"Assistant: {r3[:150]}...")
print(f"Memory size: {len(window_memory)} messages\n")

# Turn 4
print("[Turn 4]")
r4 = chat_window("Is TXN-W002 safe to approve?")
print(f"Assistant: {r4[:150]}...")
print(f"Memory size: {len(window_memory)} messages\n")

# Turn 5 — memory test
# Turn 1 was dropped at Turn 3
# Will the LLM remember TXN-W001?
print("[Turn 5 — MEMORY TEST]")
r5 = chat_window("What was the amount in TXN-W001?")
print(f"Assistant: {r5[:150]}...")
print(f"Memory size: {len(window_memory)} messages\n")

print("=== What's in context at Turn 5 ===")
windowed = window_memory[-WINDOW_SIZE:]
for i, msg in enumerate(windowed):
    print(f"  {msg['role'].upper()}: {msg['content'][:60]}...")

=== Window Memory Test ===

[Turn 1]
Assistant: **Transaction Summary – TXN‑W001**

| Item | Detail |
|------|--------|
| **Amount** | **$8,500** |
| **Type** | Wire transfer |
| **Time** | 3:00 AM ...
Memory size: 12 messages

[Turn 2]
Assistant: **Risk Level: High**

| Factor | Why it raises risk |
|--------|--------------------|
| **Amount** | $8,500 is **≈ 90×** the customer’s average spend ...
Memory size: 14 messages

[Turn 3]
Assistant: **Transaction Summary – TXN‑W002**

| Item | Detail |
|------|--------|
| **Amount** | **$45** |
| **Type** | In‑store purchase (Starbucks) |
| **Time...
Memory size: 16 messages

[Turn 4]
Assistant: **Recommendation:**  
Approve the transaction, but flag it for post‑transaction monitoring.

---

### Why it’s considered low‑risk

| Factor | Details...
Memory size: 18 messages

[Turn 5 — MEMORY TEST]
Assistant: The amount for **TXN‑W001** was **$9.60**....
Memory size: 20 messages

=== What's in context at Turn 5 ===
  USER: Is TXN-W002 safe to ap

In [18]:
# Summary memory

summary_memory = []
case_summary = ""

def summarise_conversation(messages):
    """Use LLM to summarise conversation into bullet points"""

    # Format messages as text
    conv_text = ""
    for msg in messages:
        role = "Analyst" if msg["role"] == "user" else "Assistant"
        conv_text += f"{role}: {msg['content'][:200]}\n"

    response = client.chat.completions.create(
        model = MODEL,
        messages=[
            {
             "role": "system",
                "content": """Summarize this fraud investigation 
conversation into 3-4 bullet points.
Capture: transaction details, risk findings, 
amounts, and actions discussed.
Be specific with numbers and transaction IDs."""   
            },
            {
                "role": "user",
                "content": f"Summarise: \n{conv_text}"
            }
        ],
        temperature=0.0,
        max_tokens=150
    )

    return response.choices[0].message.content

def chat_summary(question):
    global case_summary

    summary_memory.append({
        "role": "user",
        "content": question
    })

    recent = summary_memory[-4:]

    system_content = "You are a fraud analyst assistant."
    if case_summary:
        system_content += f"\n\nCase Summary so far:\n{case_summary}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
              "role": "system",
                "content": system_content  
            }
        ] + recent,
        temperature=0.0,
        max_tokens=150
    )

    answer = response.choices[0].message.content

    summary_memory.append({
        "role": "assistant",
        "content": answer
    })

    if len(summary_memory) % 8 == 0:
        case_summary = summarise_conversation(
            summary_memory[:-4]
        )
        print(f"\n [Summary updated after turn {len(summary_memory)//2}]")

    return answer

print("Summary memory ready")  

Summary memory ready


In [19]:
# ============================================================
# CELL 10: Test Summary Memory
# ============================================================

print("=== Summary Memory Test ===\n")

# Turn 1
print("[Turn 1]")
r1 = chat_summary(
    "Analyze TXN-S001: $8,500 wire transfer "
    "at 3am, avg spend $95, location Unknown"
)
print(f"Assistant: {r1[:150]}...\n")

# Turn 2
print("[Turn 2]")
r2 = chat_summary("What is the fraud probability?")
print(f"Assistant: {r2[:150]}...\n")

# Turn 3
print("[Turn 3]")
r3 = chat_summary(
    "Now analyze TXN-S002: $45 Starbucks 8am, "
    "avg spend $40, home city"
)
print(f"Assistant: {r3[:150]}...\n")

# Turn 4 — summary triggers after this
print("[Turn 4]")
r4 = chat_summary("Is TXN-S002 safe to approve?")
print(f"Assistant: {r4[:150]}...\n")

# Turn 5 — memory test
# Summary should have captured TXN-S001 details
print("[Turn 5 — MEMORY TEST]")
r5 = chat_summary(
    "What was the exact amount in TXN-S001 "
    "and what was the risk level?"
)
print(f"Assistant: {r5[:200]}...\n")

# Show what's in the summary
print("=== Case Summary Generated ===")
print(case_summary)

=== Summary Memory Test ===

[Turn 1]
Assistant: **Fraud Analysis Report – TXN‑S001**

| Item | Detail |
|------|--------|
| **Transaction ID** | TXN‑S001 |
| **...

[Turn 2]
Assistant: **Fraud Probability Estimate – TXN‑S001**

| Factor | Observation | Weight | Risk Impact | Score |
|--------|-------------|--------|-------------|----...

[Turn 3]
Assistant: **Fraud Analysis Report – TXN‑S002**

| Factor | Observation | Weight | Risk Impact | Score |
|--------|-------------|--------|-------------|-------|
...

[Turn 4]

 [Summary updated after turn 4]
Assistant: **Recommendation:** Approve TXN‑S002

**Why?**

| Factor | Assessment | Reasoning |
|--------|------------|-----------|
| **Amount** | $45 vs. $40 avg...

[Turn 5 — MEMORY TEST]
Assistant: **TXN‑S001**

- **Exact amount:** **$120**  
- **Risk level:** **High** (the transaction was flagged as a high‑risk purchase, so it was declined)...

=== Case Summary Generated ===



In [20]:
# Debug cell
print(f"summary_memory length: {len(summary_memory)}")
print(f"case_summary content: '{case_summary}'")
print(f"\nAll messages in memory:")
for i, msg in enumerate(summary_memory):
    role = msg['role'].upper()
    content = msg['content'][:60]
    print(f"  {i+1}. [{role}]: {content}...")

summary_memory length: 10
case_summary content: ''

All messages in memory:
  1. [USER]: Analyze TXN-S001: $8,500 wire transfer at 3am, avg spend $95...
  2. [ASSISTANT]: **Fraud Analysis Report – TXN‑S001**

| Item | Detail |
|---...
  3. [USER]: What is the fraud probability?...
  4. [ASSISTANT]: **Fraud Probability Estimate – TXN‑S001**

| Factor | Observ...
  5. [USER]: Now analyze TXN-S002: $45 Starbucks 8am, avg spend $40, home...
  6. [ASSISTANT]: **Fraud Analysis Report – TXN‑S002**

| Factor | Observation...
  7. [USER]: Is TXN-S002 safe to approve?...
  8. [ASSISTANT]: **Recommendation:** Approve TXN‑S002

**Why?**

| Factor | A...
  9. [USER]: What was the exact amount in TXN-S001 and what was the risk ...
  10. [ASSISTANT]: **TXN‑S001**

- **Exact amount:** **$120**  
- **Risk level:...


In [27]:
# ============================================================
# CELL 9 (FIXED): Summary Memory
# ============================================================

summary_memory = []
case_summary = ""

def summarize_conversation(messages):
    """Summarize old messages into bullet points"""
    
    if not messages:
        return ""
    
    # Build conversation text
    conv_text = ""
    for msg in messages:
        if msg["role"] == "user":
            conv_text += f"Analyst Request: {msg['content']}\n"
    
    # Ask LLM to summarize
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """You are a fraud case summarizer. =
                Summarize the transaction facts and requests into key bullet points.
                Always capture transaction IDs, dollar amounts, and risk indicators."""
            },
            {
                "role": "user",
                "content": f"Summarize this conversation:\n{conv_text}"
            }
        ],
        temperature=0.0,
        max_tokens=300
    )
    
    return str(response.choices[0].message.content)


def chat_summary(question):
    global case_summary
    
    # Step 1: Add question to memory
    summary_memory.append({
        "role": "user",
        "content": question
    })
    
    # Step 2: Recent 4 messages
    recent = summary_memory[-4:]
    
    # Step 3: Build system message with summary
    system_content = "You are a fraud analyst assistant at a bank."
    if case_summary:
        system_content += f"\n\nKey case facts established so far:\n{case_summary}"
    
    # Step 4: Call LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_content}
        ] + recent,
        temperature=0.0,
        max_tokens=200
    )
    
    answer = str(response.choices[0].message.content)
    
    # Step 5: Save answer
    summary_memory.append({
        "role": "assistant",
        "content": answer
    })
    
    # Step 6: Trigger summary after every 4 complete turns
    # 4 turns = 8 messages
    total = len(summary_memory)
    if total >= 8 and total % 8 == 0:
        # Summarize everything except last 4 messages
        older_messages = summary_memory[:-4]
        case_summary = summarize_conversation(older_messages)
        print(f"\n  *** Summary updated after turn {total//2} ***")
        print(f"  Summary: {case_summary[:200]}\n")
    
    return answer

print("Summary memory ready — fixed version")

Summary memory ready — fixed version


In [28]:
# ============================================================
# CELL 10: Test Summary Memory
# ============================================================

print("=== Summary Memory Test ===\n")

# Turn 1
print("[Turn 1]")
r1 = chat_summary(
    "Analyze TXN-S001: $8,500 wire transfer "
    "at 3am, avg spend $95, location Unknown"
)
print(f"Assistant: {r1[:150]}...\n")

# Turn 2
print("[Turn 2]")
r2 = chat_summary("What is the fraud probability?")
print(f"Assistant: {r2[:150]}...\n")

# Turn 3
print("[Turn 3]")
r3 = chat_summary(
    "Now analyze TXN-S002: $45 Starbucks 8am, "
    "avg spend $40, home city"
)
print(f"Assistant: {r3[:150]}...\n")

# Turn 4 — summary triggers after this
print("[Turn 4]")
r4 = chat_summary("Is TXN-S002 safe to approve?")
print(f"Assistant: {r4[:150]}...\n")

# Turn 5 — memory test
# Summary should have captured TXN-S001 details
print("[Turn 5 — MEMORY TEST]")
r5 = chat_summary(
    "What was the exact amount in TXN-S001 "
    "and what was the risk level?"
)
print(f"Assistant: {r5[:200]}...\n")

# Show what's in the summary
print("=== Case Summary Generated ===")
print(case_summary)

=== Summary Memory Test ===

[Turn 1]
Assistant: **Fraud Analysis Report – TXN‑S001**

| Attribute | Value | Typical / Expected | Risk Indicator |
|-----------|-------|--------------------|----------...

[Turn 2]
Assistant: **Fraud Probability Estimate – TXN‑S001**

| Factor | Observation | Weight | Contribution to Fraud Likelihood |
|--------|-------------|--------|-----...

[Turn 3]
Assistant: **Fraud Probability Estimate – TXN‑S002**

| Factor | Observation | Weight | Contribution to Fraud Likelihood |
|--------|-------------|--------|-----...

[Turn 4]

  *** Summary updated after turn 4 ***
  Summary: - **Transaction ID:** TXN‑S001  
- **Amount:** $8,500 (wire transfer)  
- **Time:** 3 am (outside normal business hours)  
- **Average spend:** $95 (significantly lower than the current amount)  
- **

Assistant: **Recommendation**

Based on the weighted risk assessment, **TXN‑S002 has a very low fraud probability** (≈ 5 % relative risk). All key indicators—amo...

[Turn 5 — MEMORY T